# DSAR × Lakeflow Declarative Pipelines · 00 · Setup, volume & INITIAL landing files

Builds the **isolated demo schema**, a **Unity Catalog volume** that is the Auto
Loader **landing zone**, writes the **initial batch of raw JSON files** into it, and
seeds the multi-subject **`dsar_request`** erasure queue.

The pipeline (`01`/`01b`) ingests these files with **Auto Loader**
(`cloudFiles`) into a `raw_user` streaming table, then masks → cleans → aggregates.

```
/Volumes/<cat>/<schema>/raw_user/            <- UC volume (landing zone)
    landing/initial/*.json                   <- 00 writes these  (initial load)
    landing/incremental/*.json               <- 00b / manual upload (Part B)
                |
          [Auto Loader cloudFiles]
                v
   raw_user (streaming) -> bronze (mask) -> silver -> gold (MV)
```

> **Why files, not a Delta table?** Real ingest (Allegiant's Merlot) lands *files*
> that Auto Loader picks up. For CCPA that matters: erasing the *tables* is not
> enough — the cleartext PII also sits in these **source files**, and a **full
> refresh re-reads them**. So `02` scrubs the tables **and** these volume files.

> Batch notebook — **Run all** once per cycle. Idempotent: recreates the schema and
> clears the landing folders. The **volume itself is reused** (`IF NOT EXISTS`) so its
> governance tags / RemoveAfter are preserved.


## 0. Configuration


In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema", "allegiant_air_sdp_dsar", "2 Schema (isolated demo)")
dbutils.widgets.text("volume", "raw_user", "3 Landing volume name")
dbutils.widgets.text("num_users", "2000", "4 Number of customers")
dbutils.widgets.text("events_per_user", "5", "5 Raw events per customer")
dbutils.widgets.text("num_files", "8", "6 Initial JSON files to write")
dbutils.widgets.text("num_requests", "10", "7 DSAR requests to seed (wave 1)")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()
VOLUME  = dbutils.widgets.get("volume").strip()
FQ      = f"{CATALOG}.{SCHEMA}"
N_USERS = int(dbutils.widgets.get("num_users"))
N_EV    = int(dbutils.widgets.get("events_per_user"))
N_FILES = int(dbutils.widgets.get("num_files"))
N_REQ   = int(dbutils.widgets.get("num_requests"))

VOL_ROOT    = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
LANDING     = f"{VOL_ROOT}/landing"
INITIAL     = f"{LANDING}/initial"
INCREMENTAL = f"{LANDING}/incremental"
print("Schema:", FQ, "| volume:", VOLUME, "| customers:", N_USERS, "| events:", N_USERS*N_EV)
print("Landing (initial):", INITIAL)


## 1. Schema + landing volume

The schema is rebuilt clean each run. The **volume is reused** if it already exists
(preserves its tags); we only clear the landing subfolders so re-runs start fresh.


In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"DROP SCHEMA IF EXISTS {FQ} CASCADE")
spark.sql(f"CREATE SCHEMA {FQ} COMMENT 'DSAR erasure integrated with a Lakeflow Declarative Pipeline (Auto Loader)'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQ}.{VOLUME} COMMENT 'Auto Loader landing zone for raw_user events'")

# clear landing folders for a clean run (volume itself is preserved)
import os
for d in (INITIAL, INCREMENTAL):
    try: dbutils.fs.rm(d, recurse=True)
    except Exception: pass
    dbutils.fs.mkdirs(d)
print("Ready. Landing folders reset under", LANDING)


## 2. Generate the INITIAL raw customer events (in memory)

Same schema as before — cleartext PII (`email`, `full_name`, nested `profile_json`)
plus the stable non-PII `user_id`. We'll write these as JSON files for Auto Loader.


In [ ]:
from pyspark.sql import functions as F

FIRST = ["Alex","Sam","Jordan","Taylor","Morgan","Casey","Riley","Jamie","Drew","Quinn"]
LAST  = ["Lucero","Ortiz","Nguyen","Patel","Kim","Diaz","Reed","Cole","Shah","Vega"]

base = (
    spark.range(N_USERS)
    .withColumn("user_id", F.concat(F.lit("U"), F.lpad(F.col("id").cast("string"), 6, "0")))
    .withColumn("first", F.element_at(F.array(*[F.lit(x) for x in FIRST]), (F.col("id") % 10 + 1).cast("int")))
    .withColumn("last",  F.element_at(F.array(*[F.lit(x) for x in LAST]),  (F.col("id") % 10 + 1).cast("int")))
    .withColumn("full_name", F.concat_ws(" ", "first", "last"))
    .withColumn("email", F.concat_ws("", F.lower("first"), F.lit("."), F.lower("last"),
                                     F.col("id").cast("string"), F.lit("@example.com")))
)
events = (
    base.withColumn("evt", F.explode(F.sequence(F.lit(0), F.lit(N_EV - 1))))
    .withColumn("event_id", F.concat_ws("-", "user_id", F.col("evt").cast("string")))
    .withColumn("revenue", F.round(F.rand(7) * 500, 2))
    .withColumn("event_ts", F.expr("current_timestamp() - make_interval(0,0,0,cast(evt as int),0,0,0)"))
    .withColumn("_ingest_ts", F.expr("current_timestamp() + make_interval(0,0,0,0,0,cast(evt as int),0)"))
    .withColumn("profile_json", F.to_json(F.struct(
        F.struct(F.col("email").alias("email"), F.col("full_name").alias("name")).alias("contact"),
        F.struct(F.lit("gold").alias("tier"), F.col("revenue").alias("ltv")).alias("loyalty"))))
    .select("event_id", "user_id", "email", "full_name", "profile_json",
            "revenue",
            F.date_format("event_ts", "yyyy-MM-dd'T'HH:mm:ss").alias("event_ts"),
            F.date_format("_ingest_ts", "yyyy-MM-dd'T'HH:mm:ss").alias("_ingest_ts"))
)
print("Generated events:", events.count())


## 3. Write the initial batch as JSON files into the volume

`num_files` JSON part-files land in `landing/initial/`. Auto Loader will pick these
up on the pipeline's first run.


In [ ]:
(events.repartition(N_FILES)
        .write.format("json").mode("overwrite")
        .save(INITIAL))

files = [f for f in dbutils.fs.ls(INITIAL) if f.name.endswith(".json")]
print(f"Wrote {len(files)} JSON file(s) to {INITIAL}:")
for f in files[:10]:
    print(f"  {f.name:<60} {f.size:>10} bytes")
print("\nSample record:")
display(spark.read.json(INITIAL).limit(3))


## 4. `dsar_request` — the erasure queue (multi-subject, 1st wave)

Each row names one subject by **email** with a `request_type` (DELETE or OBFUSCATE)
and a `status`. Notebook `02` processes all PENDING rows — erasing every named
subject across every layer **and** scrubbing the volume source files.


In [ ]:
from pyspark.sql import functions as F

# pick N_REQ distinct real subjects from the landing files (email = intake key)
subjects = (spark.read.json(INITIAL).select("email").distinct()
            .orderBy("email").limit(N_REQ).collect())
emails = [r["email"] for r in subjects]

# alternate DELETE / OBFUSCATE so both modes are exercised
rows = [
    (f"REQ-{i+1:03d}", e, ("DELETE" if i % 2 == 0 else "OBFUSCATE"), "PENDING")
    for i, e in enumerate(emails)
]
df = spark.createDataFrame(rows, "request_id string, subject_email string, request_type string, status string") \
          .withColumn("request_date", F.current_date()) \
          .withColumn("deadline_date", F.date_add(F.current_date(), 45))
df.write.mode("overwrite").saveAsTable(f"{FQ}.dsar_request")

print("Seeded DSAR requests (wave 1):")
display(spark.table(f"{FQ}.dsar_request"))
print(f"\nSeeded {len(emails)} request(s). Demo subjects (used by 02):", emails)


## 5. Next

1. **`01_sdp_pipeline`** (or `01b_sdp_pipeline_cdc_variant`) — attach as a Lakeflow
   pipeline source. Set config `dsar.catalog` / `dsar.schema` (and `dsar.volume` if
   you changed it). **Start** → Auto Loader ingests `landing/initial/*.json` into
   `raw_user`, then bronze/silver/gold.
2. **`02_erasure`** — process the DSAR queue: erase every subject at every layer
   **and scrub the volume files**, then refresh gold.
3. **`00b_incremental_landing`** — Part B: drop new files + a 2nd erasure wave.

See `usage.md` for the full runbook.
